## Transformer

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # Shape: [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [batch_size, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]


class SignalTransformerClassifier(nn.Module):
    def __init__(
        self, 
        input_dim: int,          # Num of continuous features per step (e.g., 4 for p_x, p_y, p_z, E)
        d_model: int = 128,      # Internal transformer hidden dimension
        nhead: int = 8,          # Multi-head attention heads
        num_layers: int = 4,     # Number of TransformerEncoder layers
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        use_pos_encoding: bool = True
    ):
        super().__init__()
        self.use_pos_encoding = use_pos_encoding

        # 1. Continuous Feature Projection (Replaces lookup table / nn.Embedding)
        self.input_projection = nn.Linear(input_dim, d_model)

        # 2. Learnable [CLS] token to aggregate global event context
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # 3. Positional Encoding
        if self.use_pos_encoding:
            self.pos_encoder = SinusoidalPositionalEncoding(d_model)

        # 4. Standard Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Inputs shape: [batch, seq, feature]
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Classification Head (Signal vs Background)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)  # Binary output logits
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        x: [batch_size, seq_len, input_dim]
        mask: Optional boolean tensor [batch_size, seq_len] where True indicates padded elements.
        """
        batch_size = x.size(0)

        # Step 1: Project continuous inputs to hidden dimension
        x = self.input_projection(x)  # [batch_size, seq_len, d_model]

        # Step 2: Add Positional Encoding (if sequence order matters)
        if self.use_pos_encoding:
            x = self.pos_encoder(x)

        # Step 3: Prepend [CLS] token to each sequence
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [batch_size, 1, d_model]
        x = torch.cat((cls_tokens, x), dim=1)                   # [batch_size, seq_len + 1, d_model]

        # Step 4: Adjust padding mask if sequence length varies across the batch
        if mask is not None:
            # Prepend False to key_padding_mask so [CLS] is never masked out
            cls_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=mask.device)
            mask = torch.cat((cls_mask, mask), dim=1)

        # Step 5: Pass through Transformer
        output = self.transformer_encoder(x, src_key_padding_mask=mask)

        # Step 6: Extract the [CLS] representation (first token)
        cls_representation = output[:, 0, :]  # [batch_size, d_model]

        # Step 7: Classification Logits
        logits = self.classifier(cls_representation)  # [batch_size, 1]
        return logits.squeeze(-1)


# --- Example Usage ---
if __name__ == "__main__":
    # Parameters
    batch_size = 32
    seq_len = 50         # e.g., 50 time steps or 50 particle hits
    input_dim = 4        # e.g., 4 continuous features per step [p_x, p_y, p_z, E]

    # Dummy continuous signal event batch
    events = torch.randn(batch_size, seq_len, input_dim)

    # Initialize model (Set use_pos_encoding=False if order is irrelevant)
    model = SignalTransformerClassifier(
        input_dim=input_dim,
        d_model=128,
        use_pos_encoding=True  # Set False for unordered particle sets
    )

    # Forward pass
    logits = model(events)  # Raw logits for Binary Cross Entropy Loss (BCEWithLogitsLoss)
    probabilities = torch.sigmoid(logits)  # Signal probability per event

    print(f"Input shape: {events.shape}")
    print(f"Output probability shape: {probabilities.shape}")